In [1]:
import logging
import os
import sys
import pandas as pd

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(name)s | %(message)s",
    handlers=[logging.StreamHandler(sys.stdout)],
    force=True,
)

DATA_DIR = os.environ.get("DATA_DIR", "../data")
impressions_path = os.path.join(DATA_DIR, "impressions.json")
clicks_path = os.path.join(DATA_DIR, "clicks.json")

In [2]:
from pipeline.session import get_spark
from pipeline.io.loader import load
from pipeline.constants import IMPRESSIONS_SCHEMA, CLICKS_SCHEMA

In [3]:
import pyspark.sql.functions as f
from pyspark.sql.types import StringType, IntegerType, StructType, StructField, FloatType
from pyspark.sql.window import Window

In [4]:
spark = get_spark(app_name="EDA")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/04/14 19:17:27 WARN Utils: Your hostname, ilya, resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/04/14 19:17:27 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/14 19:17:38 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


### Research schemas:

In [5]:
impressions_df = pd.read_json(impressions_path)

print(impressions_df.info())
print("------------------------------------------------------------------------------------------------------------")
print(impressions_df.head())

<class 'pandas.DataFrame'>
RangeIndex: 1038 entries, 0 to 1037
Data columns (total 5 columns):
 #   Column         Non-Null Count  Dtype
---  ------         --------------  -----
 0   id             1038 non-null   str  
 1   user_id        1034 non-null   str  
 2   app_id         1038 non-null   int64
 3   country_code   1033 non-null   str  
 4   advertiser_id  1038 non-null   int64
dtypes: int64(2), str(3)
memory usage: 40.7 KB
None
------------------------------------------------------------------------------------------------------------
                                     id                               user_id  \
0  ca68ca05-a207-4a66-b1d0-93ce9dbe8a8d  e1173a98-ba39-4818-87c2-7e12537947af   
1  1337c971-b6c4-4e9d-9a93-83e0860fb513  c17e3706-d0d5-4c5e-9986-fcb6d779699c   
2  ea7c57cd-a8c9-4cee-ac75-61a479c98e26  28670b09-e02a-43ae-812b-4fae9859f0f4   
3  9ad7a12a-e214-4e8f-ad7e-49158a038e48  0450f615-eb99-4e8b-b50a-a36c7a622630   
4  494ca27e-542e-4e74-b1c2-c14fbd955f82  a2f7

In [6]:
impressions_schema = StructType([
    StructField("id", StringType(), nullable=False),
    StructField("user_id", StringType(), nullable=False),
    StructField("app_id", IntegerType(), nullable=False),
    StructField("country_code", StringType(), nullable=False),
    StructField("advertiser_id", IntegerType(), nullable=False),
])

impressions_df = spark.read.option("multiLine", True).format("json").schema(impressions_schema).load(impressions_path)
impressions_df.show(5)
impressions_df.printSchema()

+--------------------+--------------------+------+------------+-------------+
|                  id|             user_id|app_id|country_code|advertiser_id|
+--------------------+--------------------+------+------------+-------------+
|ca68ca05-a207-4a6...|e1173a98-ba39-481...|     1|          US|            8|
|1337c971-b6c4-4e9...|c17e3706-d0d5-4c5...|     1|          US|            8|
|ea7c57cd-a8c9-4ce...|28670b09-e02a-43a...|     1|          GB|            1|
|9ad7a12a-e214-4e8...|0450f615-eb99-4e8...|     3|          US|           13|
|494ca27e-542e-4e7...|a2f77409-c73d-4ef...|     2|          MX|           13|
+--------------------+--------------------+------+------------+-------------+
only showing top 5 rows
root
 |-- id: string (nullable = true)
 |-- user_id: string (nullable = true)
 |-- app_id: integer (nullable = true)
 |-- country_code: string (nullable = true)
 |-- advertiser_id: integer (nullable = true)



In [7]:
clicks_df = pd.read_json(clicks_path)

print(clicks_df.info())
print("------------------------------------------------------------------------------------------------------------")
print(clicks_df.head())

<class 'pandas.DataFrame'>
RangeIndex: 705 entries, 0 to 704
Data columns (total 3 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   id             705 non-null    str    
 1   impression_id  705 non-null    str    
 2   revenue        704 non-null    float64
dtypes: float64(1), str(2)
memory usage: 16.7 KB
None
------------------------------------------------------------------------------------------------------------
                                     id                         impression_id  \
0  a1890d10-a7a0-40c8-81bd-b33a24e3e370  15a5601a-cc01-4b5c-ace7-2e27939c2994   
1  0ef2ef8c-ad37-44e7-a98b-68c8b3090229  b2c4dfba-9c40-47be-99ff-c6a073cf158c   
2  c97eac3e-6de0-48de-ae7e-bd65624de124  a05f83ed-90ed-4d90-a1ef-82b30e22772c   
3  90ac3a4f-6a3d-4fbf-a610-1b60b73df787  4d55efa0-f189-4e2a-be52-cb660df4e89e   
4  218f41e7-1efd-4344-9107-623ad9435072  b9d50050-8d1a-49e9-99b1-8bce47fcc2aa   

   revenue  
0     3.19  
1     2.32 

In [8]:
clicks_schema = StructType([
    StructField("id", StringType(), nullable=False),
    StructField("impression_id", StringType(), nullable=False),
    StructField("revenue", FloatType(), nullable=False),
])

clicks_df = spark.read.option("multiLine", True).format("json").schema(clicks_schema).load(clicks_path)
clicks_df.show(5)
clicks_df.printSchema()

+--------------------+--------------------+-------+
|                  id|       impression_id|revenue|
+--------------------+--------------------+-------+
|a1890d10-a7a0-40c...|15a5601a-cc01-4b5...|   3.19|
|0ef2ef8c-ad37-44e...|b2c4dfba-9c40-47b...|   2.32|
|c97eac3e-6de0-48d...|a05f83ed-90ed-4d9...|   0.76|
|90ac3a4f-6a3d-4fb...|4d55efa0-f189-4e2...|   2.55|
|218f41e7-1efd-434...|b9d50050-8d1a-49e...|   0.58|
+--------------------+--------------------+-------+
only showing top 5 rows
root
 |-- id: string (nullable = true)
 |-- impression_id: string (nullable = true)
 |-- revenue: float (nullable = true)



### Check duplicates in joined dataframe:

In [9]:
df = load(spark, impressions_path, IMPRESSIONS_SCHEMA, clicks_path, CLICKS_SCHEMA)

In [10]:
df.show(3)

+--------------------+--------------------+------+------------+-------------+--------------------+-------+
|       impression_id|             user_id|app_id|country_code|advertiser_id|            click_id|revenue|
+--------------------+--------------------+------+------------+-------------+--------------------+-------+
|ca68ca05-a207-4a6...|e1173a98-ba39-481...|     1|          US|            8|cecfe757-214b-489...|   3.46|
|1337c971-b6c4-4e9...|c17e3706-d0d5-4c5...|     1|          US|            8|ec9ac8b4-6855-465...|   1.75|
|ea7c57cd-a8c9-4ce...|28670b09-e02a-43a...|     1|          GB|            1|8f9f6bb0-ba7d-49e...|    0.1|
+--------------------+--------------------+------+------------+-------------+--------------------+-------+
only showing top 3 rows


In [11]:
(
    df.groupBy("impression_id")
    .agg(f.count("*").alias("cnt"))
    .orderBy(f.desc("cnt"))
    .show(n=5, truncate=False)
)

+------------------------------------+---+
|impression_id                       |cnt|
+------------------------------------+---+
|504a9c68-6344-4a89-9c00-fe01a4f890e6|4  |
|0d72a2fd-c030-4364-b21f-561b68ef73d9|4  |
|ac3e49f2-2dca-431c-bbf9-6a4d4bb35cf9|4  |
|ecfd63f0-5491-4674-ac51-9fdfd9de51e0|4  |
|053c83fa-8f41-4658-8ee7-0cc136d61358|4  |
+------------------------------------+---+
only showing top 5 rows


In [12]:
df.filter(f.col("impression_id") == "504a9c68-6344-4a89-9c00-fe01a4f890e6").show(n=5, truncate=False)

+------------------------------------+------------------------------------+------+------------+-------------+--------+-------+
|impression_id                       |user_id                             |app_id|country_code|advertiser_id|click_id|revenue|
+------------------------------------+------------------------------------+------+------------+-------------+--------+-------+
|504a9c68-6344-4a89-9c00-fe01a4f890e6|9e3b4225-6e96-4159-abaa-35850945427d|4     |DE          |1            |NULL    |NULL   |
|504a9c68-6344-4a89-9c00-fe01a4f890e6|9e3b4225-6e96-4159-abaa-35850945427d|4     |DE          |1            |NULL    |NULL   |
|504a9c68-6344-4a89-9c00-fe01a4f890e6|9e3b4225-6e96-4159-abaa-35850945427d|4     |DE          |1            |NULL    |NULL   |
|504a9c68-6344-4a89-9c00-fe01a4f890e6|9e3b4225-6e96-4159-abaa-35850945427d|4     |DE          |1            |NULL    |NULL   |
+------------------------------------+------------------------------------+------+------------+-------------+--

We see duplicates in the joined DataFrame when grouping by impression_id.

In [13]:
(
    df.groupBy("click_id")
    .agg(f.count("*").alias("cnt"))
    .orderBy(f.desc("cnt"))
    .show(n=5, truncate=False)
)

+------------------------------------+---+
|click_id                            |cnt|
+------------------------------------+---+
|NULL                                |331|
|2add46d1-e846-4566-a4f4-4b99f596a75b|4  |
|82f8185a-d034-4e12-90d8-7d520573660e|4  |
|9f102b25-7660-4900-a38a-b1e653ab631e|4  |
|9d03b90b-b5e4-462e-a67e-2be56c0f1f98|4  |
+------------------------------------+---+
only showing top 5 rows


In [14]:
df.filter(f.col("click_id") == "2add46d1-e846-4566-a4f4-4b99f596a75b").show(n=5, truncate=False)

+------------------------------------+------------------------------------+------+------------+-------------+------------------------------------+-------+
|impression_id                       |user_id                             |app_id|country_code|advertiser_id|click_id                            |revenue|
+------------------------------------+------------------------------------+------+------------+-------------+------------------------------------+-------+
|974f9aeb-e6cd-4d36-af9c-5e1624efcfb0|e55035aa-6348-4043-a4a9-b7c2b2401070|2     |MX          |17           |2add46d1-e846-4566-a4f4-4b99f596a75b|0.69   |
|974f9aeb-e6cd-4d36-af9c-5e1624efcfb0|e55035aa-6348-4043-a4a9-b7c2b2401070|2     |MX          |17           |2add46d1-e846-4566-a4f4-4b99f596a75b|0.69   |
|974f9aeb-e6cd-4d36-af9c-5e1624efcfb0|e55035aa-6348-4043-a4a9-b7c2b2401070|2     |MX          |17           |2add46d1-e846-4566-a4f4-4b99f596a75b|0.69   |
|974f9aeb-e6cd-4d36-af9c-5e1624efcfb0|e55035aa-6348-4043-a4a9-b7c2b240

We also see duplicates in the joined DataFrame when grouping by `click_id`, so I need to deduplicate the data right after reading it.

In [15]:
df_new = df.distinct()

(
    df_new.groupBy("click_id")
    .agg(f.count("*").alias("cnt"))
    .orderBy(f.desc("cnt"))
    .show(n=5, truncate=False)
)

+------------------------------------+---+
|click_id                            |cnt|
+------------------------------------+---+
|NULL                                |312|
|6a5ebf8a-e1c2-4b3c-823a-2913c0b71082|1  |
|7de31e1d-d944-4928-b2dd-45c74eaffe0e|1  |
|b940d07c-ff3e-4be8-904b-26020f3f0d05|1  |
|4bd5d93e-c661-42a4-8673-cee3175983e3|1  |
+------------------------------------+---+
only showing top 5 rows


### Check NULL and strange values in fields:

In [22]:
df_new.select("country_code").distinct().toPandas()["country_code"].to_list()

['FR',
 'AU',
 'MX',
 'CA',
 'IN',
 'GB',
 'JP',
 'DE',
 'US',
 'XX',
 'BR',
 'ZZZ',
 '??',
 nan]

As we can see, country_code values are in the ISO alpha‑2 country code format. However, there are several unusual values that are not valid country codes in the usual sense. For example, ZZZ is unexpected because it does not exist in either the alpha‑2 or alpha‑3 standards, and values like NAN and ?? are also invalid. Therefore, I decided to group all such values into a single Unknown category.

In [30]:
# values in alpha2 country codes
country_codes_alpha2 = [
    "AF", "AX", "AL", "DZ", "AS", "AD", "AO", "AI", "AQ", "AG", "AR", "AM",
    "AW", "AU", "AT", "AZ", "BS", "BH", "BD", "BB", "BY", "BE", "BZ", "BJ",
    "BM", "BT", "BO", "BQ", "BA", "BW", "BV", "BR", "IO", "BN", "BG", "BF",
    "BI", "CV", "KH", "CM", "CA", "KY", "CF", "TD", "CL", "CN", "CX", "CC",
    "CO", "KM", "CD", "CG", "CK", "CR", "CI", "HR", "CU", "CW", "CY", "CZ",
    "DK", "DJ", "DM", "DO", "EC", "EG", "SV", "GQ", "ER", "EE", "SZ", "ET",
    "FK", "FO", "FJ", "FI", "FR", "GF", "PF", "TF", "GA", "GM", "GE", "DE",
    "GH", "GI", "GR", "GL", "GD", "GP", "GU", "GT", "GG", "GN", "GW", "GY",
    "HT", "HM", "VA", "HN", "HK", "HU", "IS", "IN", "ID", "IR", "IQ", "IE",
    "IM", "IL", "IT", "JM", "JP", "JE", "JO", "KZ", "KE", "KI", "KP", "KR",
    "KW", "KG", "LA", "LV", "LB", "LS", "LR", "LY", "LI", "LT", "LU", "MO",
    "MG", "MW", "MY", "MV", "ML", "MT", "MH", "MQ", "MR", "MU", "YT", "MX",
    "FM", "MD", "MC", "MN", "ME", "MS", "MA", "MZ", "MM", "NA", "NR", "NP",
    "NL", "NC", "NZ", "NI", "NE", "NG", "NU", "NF", "MK", "MP", "NO", "OM",
    "PK", "PW", "PS", "PA", "PG", "PY", "PE", "PH", "PN", "PL", "PT", "PR",
    "QA", "RE", "RO", "RU", "RW", "BL", "SH", "KN", "LC", "MF", "PM", "VC",
    "WS", "SM", "ST", "SA", "SN", "RS", "SC", "SL", "SG", "SX", "SK", "SI",
    "SB", "SO", "ZA", "GS", "SS", "ES", "LK", "SD", "SR", "SJ", "SE", "CH",
    "SY", "TW", "TJ", "TZ", "TH", "TL", "TG", "TK", "TO", "TT", "TN", "TR",
    "TM", "TC", "TV", "UG", "UA", "AE", "GB", "US", "UY", "UZ", "VU", "VE",
    "VN", "VG", "VI", "WF", "EH", "YE", "ZM", "ZW",
]

In [31]:
df_new = df_new.withColumn(
    "country_code",
    f.when(f.col("country_code").isin(country_codes_alpha2), f.col("country_code"))
     .otherwise(f.lit("Unknown"))
)

### Creating metrics and optimization:

#### Metrics calculation:

In [32]:
(
    df_new.groupBy("app_id", "country_code")
    .agg(f.count("*").alias("cnt"))
    .orderBy(f.desc("cnt"))
    .show(n=5, truncate=False)
)

+------+------------+---+
|app_id|country_code|cnt|
+------+------------+---+
|1     |US          |151|
|2     |US          |81 |
|1     |CA          |75 |
|3     |US          |48 |
|1     |GB          |44 |
+------+------------+---+
only showing top 5 rows


On this subset we do not observe noticeable data skew (US/CA ratio ~2:1).
However, as data volume grows, US bucket may dominate significantly.

Action threshold: if any single country_code exceeds ~5x the median partition
size, apply salting on country_code using app_id as salt source:
  salted_key = concat(country_code, '_', hash(app_id) % N)
  where N = number of salt buckets (start with 10).

Note: since we use count() (not countDistinct), partial aggregation
per salt bucket is safe — just sum the partial counts in a second pass.

#### Top-5 `advertiser_id` per app + country pair

In [33]:
advertiser_stats = (
        df_new.groupBy("app_id", "country_code", "advertiser_id").agg(
            f.count("impression_id").alias("impressions"),
            f.sum("revenue").alias("total_revenue"),
        )
    )

advertiser_stats.filter(f.col("impressions") < 5).count(), advertiser_stats.filter(f.col("impressions") >= 5).count()

(470, 36)

In [34]:
advertiser_stats = advertiser_stats.filter(advertiser_stats.impressions >= 5)

In [35]:
(
    advertiser_stats.groupBy("app_id", "country_code")
    .agg(f.collect_list("advertiser_id").alias("recommended_advertiser_ids"))
    .show(truncate=False)
)

+------+------------+----------------------------------------------------------+
|app_id|country_code|recommended_advertiser_ids                                |
+------+------------+----------------------------------------------------------+
|1     |US          |[1, 15, 3, 14, 19, 17, 7, 16, 9, 11, 18, 8, 10, 4, 13, 20]|
|2     |US          |[7, 2, 15, 13, 3, 5, 18, 19, 14, 9]                       |
|3     |US          |[18, 13]                                                  |
|2     |CA          |[20]                                                      |
|1     |CA          |[15, 8, 16, 14, 11, 2, 10]                                |
+------+------------+----------------------------------------------------------+



In [36]:
result = (
    advertiser_stats
    .withColumn("rank_struct", f.struct(f.col("total_revenue"), f.col("advertiser_id")))
    .groupBy("app_id", "country_code")
    .agg(f.sort_array(f.collect_list("rank_struct"), asc=False).alias("pairs"))
    .withColumn("pairs", f.slice(f.col("pairs"), 1, 5))
    .withColumn(
        "recommended_advertiser_ids",
        f.transform(f.col("pairs"), lambda x: x["advertiser_id"])
    )
    .drop("pairs")
)

result.show(n=1000)

+------+------------+--------------------------+
|app_id|country_code|recommended_advertiser_ids|
+------+------------+--------------------------+
|     1|          US|       [8, 18, 13, 19, 10]|
|     2|          US|         [15, 14, 5, 2, 3]|
|     3|          US|                  [13, 18]|
|     2|          CA|                      [20]|
|     1|          CA|       [16, 14, 2, 11, 15]|
+------+------------+--------------------------+



In [37]:
window = Window.partitionBy("app_id", "country_code").orderBy(f.col("total_revenue").desc())

advertiser_stats = (
    df_new.groupBy("app_id", "country_code", "advertiser_id").agg(
        f.count("impression_id").alias("impressions"),
        f.sum("revenue").alias("total_revenue"),
    )
)
advertiser_stats = advertiser_stats.filter(advertiser_stats.impressions >= 5)
advertiser_stats = advertiser_stats.withColumn("rn", f.row_number().over(window))
advertiser_stats = advertiser_stats.filter(advertiser_stats.rn <= 5)
advertiser_stats = advertiser_stats.groupBy("app_id", "country_code").agg(f.collect_list("advertiser_id").alias("recommended_advertiser_ids"))
advertiser_stats.show()

+------+------------+--------------------------+
|app_id|country_code|recommended_advertiser_ids|
+------+------------+--------------------------+
|     1|          CA|       [16, 14, 2, 11, 15]|
|     1|          US|       [8, 18, 13, 19, 10]|
|     2|          CA|                      [20]|
|     2|          US|         [15, 14, 5, 2, 3]|
|     3|          US|                  [13, 18]|
+------+------------+--------------------------+



One more time, this is also a question of optimization. In this case I prefer using ROW_NUMBER plus collect_list instead of collect_list + sort_array + slice. The reason is that collect_list on large datasets can pull all skewed US records onto a few nodes, creating huge arrays in memory, which may lead to OOM errors.

#### Median user spend

In [38]:
(
    df_new.groupBy("country_code", "user_id")
    .agg(f.sum(f.coalesce("revenue", f.lit(0.0))).alias("spend"))
    .groupBy("country_code")
    .agg(
        f.median("spend").alias("median_exact"),
        f.percentile_approx("spend", 0.5, 100).alias("approx_100"),
        f.percentile_approx("spend", 0.5, 1000).alias("approx_1000"),
        f.percentile_approx("spend", 0.5, 10000).alias("approx_10000"),
    )
    .orderBy("country_code")
    .show(truncate=False)
)

+------------+------------------+------------------+------------------+------------------+
|country_code|median_exact      |approx_100        |approx_1000       |approx_10000      |
+------------+------------------+------------------+------------------+------------------+
|AU          |1.5950000286102295|1.4900000095367432|1.4900000095367432|1.4900000095367432|
|BR          |1.8799999952316284|1.8799999952316284|1.8799999952316284|1.8799999952316284|
|CA          |1.2200000286102295|1.2000000476837158|1.2200000286102295|1.2200000286102295|
|DE          |0.8100000023841858|0.8100000023841858|0.8100000023841858|0.8100000023841858|
|FR          |0.7900000214576721|0.7900000214576721|0.7900000214576721|0.7900000214576721|
|GB          |1.309999942779541 |1.309999942779541 |1.309999942779541 |1.309999942779541 |
|IN          |0.7900000214576721|0.7900000214576721|0.7900000214576721|0.7900000214576721|
|JP          |1.5800000429153442|1.5800000429153442|1.5800000429153442|1.5800000429153442|

In [86]:
df_new.filter(f.col("country_code") == "??").show()

+--------------------+--------------------+------+------------+-------------+--------------------+-------+
|       impression_id|             user_id|app_id|country_code|advertiser_id|            click_id|revenue|
+--------------------+--------------------+------+------------+-------------+--------------------+-------+
|cf96c02d-8314-4e7...|1b1ed91a-caa4-459...|     2|          ??|           10|a50ab6dc-b0ec-4d2...|   1.89|
|ac3e49f2-2dca-431...|768b4d2c-e005-49b...|     1|          ??|            1|                NULL|   NULL|
+--------------------+--------------------+------+------------+-------------+--------------------+-------+



For most countries the approximate median (`percentile_approx`) is very close to the exact median, but for the `Unknown` bucket the distribution is heavily skewed towards zero, so the sketch used by `percentile_approx` collapses the 50th percentile to 0 while the exact median remains slightly above zero. In practice, the choice between exact and approximate median should depend on our goals: if we need precise statistics for detailed analysis or reporting, we should rely on the exact median despite its higher computational cost, whereas for large‑scale, performance‑critical workloads we can accept the approximation and ignore such minor differences for small or noisy buckets like `Unknown`.